In [ ]:
import json

import numpy as np
import tensorflow as tf
from IPython.display import HTML, display
from tensorflow.keras import callbacks, layers, models

DRIVE_DIR = "/content/drive/MyDrive"
VOCAB_PATH = f"{DRIVE_DIR}/vocab.json"
MODEL_PATH = f"{DRIVE_DIR}/gpt.keras"
START_PROMPT = "wine review : germany"
MAX_TOKENS = 80
TEMPERATURE = 0.5

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    mask = i >= j - n_src + n_dest
    mask = tf.cast(mask, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
    )
    return tf.tile(mask, mult)


class TransformerBlock(layers.Layer):
    def __init__(
        self,
        num_heads,
        key_dim,
        embed_dim,
        ff_dim,
        dropout_rate=0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        self.attn = layers.MultiHeadAttention(
            num_heads, key_dim, output_shape=embed_dim
        )
        self.dropout_1 = layers.Dropout(dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(embed_dim)
        self.dropout_2 = layers.Dropout(dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        attention_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )
        attention_output, attention_scores = self.attn(
            inputs,
            inputs,
            attention_mask=attention_mask,
            return_attention_scores=True,
        )
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)
        ffn_output = self.ffn_2(self.ffn_1(out1))
        ffn_output = self.dropout_2(ffn_output)
        return self.ln_2(out1 + ffn_output), attention_scores

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "num_heads": self.num_heads,
                "key_dim": self.key_dim,
                "embed_dim": self.embed_dim,
                "ff_dim": self.ff_dim,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(max_len, embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_len": self.max_len,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

In [ ]:
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        super().__init__()
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }
        self.top_k = top_k

    def sample_from(self, probs, temperature):
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(word, 1) for word in start_prompt.split()
        ]
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, attention = self.model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": attention[0, :, -1, :],
                }
            )
            start_tokens.append(sample_token)
            start_prompt += " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info


def print_probs(info, vocab, top_k=5):
    for item in info:
        mean_attention = np.mean(item["atts"], axis=0)
        max_attention = np.max(mean_attention)
        highlighted_text = []
        for word, score in zip(item["prompt"].split(), mean_attention):
            opacity = score / max_attention if max_attention else 0
            highlighted_text.append(
                f'<span style="background-color:rgba(135,206,250,{opacity});">'
                f"{word}</span>"
            )
        display(HTML(" ".join(highlighted_text)))

        word_probs = item["word_probs"]
        top_indices = np.argsort(word_probs)[::-1][:top_k]
        for index in top_indices:
            print(f"{vocab[index]}:\t{word_probs[index] * 100:.2f}%")
        print("--------\n")

In [ ]:
# 学習時に保存しておいた、学習時と同じ語彙を読み込む
with open(VOCAB_PATH, encoding="utf-8") as f:
    vocab = json.load(f)

In [ ]:
# gpt.save() で保存したモデルを読み込む
gpt = models.load_model(
    MODEL_PATH,
    custom_objects={
        "TransformerBlock": TransformerBlock,
        "TokenAndPositionEmbedding": TokenAndPositionEmbedding,
    },
    compile=False,
)

In [ ]:
# TextGenerator の引数はモデルではなく vocab
text_generator = TextGenerator(vocab)
text_generator.set_model(gpt)

In [ ]:
info = text_generator.generate(
    START_PROMPT, max_tokens=MAX_TOKENS, temperature=TEMPERATURE
)
print_probs(info, vocab)